# Used Car Price Prediction Notebook

Welcome! This notebook is the required starting point and submission format for the AI task.

## How this notebook works

- Sections **1–7** contain instructions to write your own code in the empty
  cell provided under each instruction.
- Section **8** is pre-written for evaluator use and **must not be modified**. We will run
  it against a **hidden test set** to score your submission.
- Your notebook must run **top-to-bottom without errors** before submission
  (`Restart Kernel & Run All`).

## 0. Setup

Standard imports and file paths are set up below.
Add any additional imports your approach needs in the same cell.


In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
)

# File paths
TRAIN_PATH = "/content/data/dataset-2.csv"        # <-- set this to your local copy of the training CSV
TARGET_COLUMN = "price"                # <-- target variable: selling price
EVALUATION_SPLIT_PATH = "./evaluation_split_task2.csv"  # organizers only — do not rely on this file existing locally

## 1. Load the Data

Load the dataset from `TRAIN_PATH` into a dataframe.

In [2]:
df = pd.read_csv(TRAIN_PATH)

print(df.shape)
display(df.head())
df.info()

(3207, 12)


,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
0,BMW,M235 i,2015,"81,000 mi.",Gasoline,320.0HP 3.0L Straight 6 Cylinder Engine Gasoli...,6-Speed M/T,White,Red,None reported,Yes,"$27,890"
1,BMW,750 i,2017,"24,650 mi.",Gasoline,445.0HP 4.4L 8 Cylinder Engine Gasoline Fuel,A/T,Silver,Black,None reported,Yes,"$50,000"
2,Aston,Martin V8 Vantage Base,2008,"62,378 mi.",Gasoline,380.0HP 4.3L 8 Cylinder Engine Gasoline Fuel,M/T,Red,Beige,At least 1 accident or damage reported,Yes,"$33,995"
3,INFINITI,QX80 Luxe,2022,"49,860 mi.",Gasoline,5.6L V8 32V GDI DOHC,Automatic,Black Obsidian,Graphite,At least 1 accident or damage reported,Yes,"$47,645"
4,Subaru,BRZ Premium,2023,"5,800 mi.",Gasoline,228.0HP 2.4L 4 Cylinder Engine Gasoline Fuel,6-Speed M/T,Black,Black,None reported,Yes,"$29,990"


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3207 entries, 0 to 3206
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   brand         3207 non-null   object
 1   model         3207 non-null   object
 2   model_year    3207 non-null   int64 
 3   milage        3207 non-null   object
 4   fuel_type     3081 non-null   object
 5   engine        3207 non-null   object
 6   transmission  3207 non-null   object
 7   ext_col       3207 non-null   object
 8   int_col       3207 non-null   object
 9   accident      3115 non-null   object
 10  clean_title   2728 non-null   object
 11  price         3207 non-null   object
dtypes: int64(1), object(11)
memory usage: 300.8+ KB


## 2. Explore and Clean the Dataset

Address data quality issues before modeling:

- Identify and handle missing / null values.
- Identify categorical variables and decide how each will be handled.
- Check for duplicate rows, inconsistent categories, and extreme price/mileage outliers.
- Document any rows/columns you drop and why.

In [3]:
def data_cleaning(df):
  df = df.copy()
    # Implement your cleaning steps here
    # Basic information

print("Dataset shape:", df.shape)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nData types:")
print(df.dtypes)

categorical_cols = df.select_dtypes(include="object").columns

print("\nCategorical columns:")
print(categorical_cols.tolist())

for col in categorical_cols:
    print(f"\n{col}: {df[col].nunique()} unique values")
    print(df[col].value_counts(dropna=False).head(10))

Dataset shape: (3207, 12)

Missing values:
brand             0
model             0
model_year        0
milage            0
fuel_type       126
engine            0
transmission      0
ext_col           0
int_col           0
accident         92
clean_title     479
price             0
dtype: int64

Duplicate rows:
0

Data types:
brand           object
model           object
model_year       int64
milage          object
fuel_type       object
engine          object
transmission    object
ext_col         object
int_col         object
accident        object
clean_title     object
price           object
dtype: object

Categorical columns:
['brand', 'model', 'milage', 'fuel_type', 'engine', 'transmission', 'ext_col', 'int_col', 'accident', 'clean_title', 'price']

brand: 56 unique values
brand
BMW              301
Ford             299
Mercedes-Benz    247
Chevrolet        235
Audi             170
Toyota           162
Porsche          155
Lexus            129
Jeep             116
Land          

In [4]:
def data_cleaning(df):
    df = df.copy()

    if "milage" in df.columns:
        df["milage"] = (
            df["milage"]
            .astype(str)
            .str.replace(",", "", regex=False)
            .str.replace(" mi.", "", regex=False)
        )
        df["milage"] = pd.to_numeric(df["milage"], errors="coerce")

    if "model_year" in df.columns:
        df["model_year"] = pd.to_numeric(
            df["model_year"],
            errors="coerce"
        )

    df = df.replace(r"^\s*$", np.nan, regex=True)

    return df

## 3. Feature Engineering

Transform the raw columns into features that are more useful for a regression model.

For example: calculating vehicle age from `model_year`, extracting horsepower/displacement from `engine`, or categorizing rare brands.

In [5]:
def feature_engineering(df):
    # Implement your feature engineering steps here

    df = df.copy()


    if "model_year" in df.columns:
        df["vehicle_age"] = 2026 - df["model_year"]

    if "engine" in df.columns:
        df["horsepower"] = (
            df["engine"]
            .astype(str)
            .str.extract(r"(\d+(?:\.\d+)?)HP")[0]
        )

        df["horsepower"] = pd.to_numeric(
            df["horsepower"],
            errors="coerce"
        )


        df["engine_size"] = (
            df["engine"]
            .astype(str)
            .str.extract(r"(\d+(?:\.\d+)?)L")[0]
        )

        df["engine_size"] = pd.to_numeric(
            df["engine_size"],
            errors="coerce"
        )

    return df

## 4. Transform, Normalize, and Scale Features

Prepare the final feature set for modeling:

- Encode categorical variables (e.g., One-Hot or Target Encoding).
- Scale/normalize continuous numerical features (e.g., `mileage`, `age`, engine specs).

In [6]:
def scale_features(df):
    # Implement your scaling/encoding steps here
    df = df.copy()
    return df

In [16]:
# Do not modify this function
def preprocess_data(df):
    df = df.copy()
    df = data_cleaning(df)
    df = feature_engineering(df)
    df = scale_features(df)
    return df

## 5. Train / Validation / Test Split

Split your cleaned data into three sets:

- **Train** — used to fit models.
- **Validation** — used for hyperparameter tuning and model comparison.
- **Test** — held out until Section 6.

In [17]:

   def split_data(df):
    # Implement your data splitting logic here
    # This function should return three dataframes: train, validation, and test

    # First split: 80% train + validation, 20% test
    train_val, test_split = train_test_split(
        df,
        test_size=0.20,
        random_state=42
    )

    # Split remaining 80% into 60% train and 20% validation
    train_split, validation_split = train_test_split(
        train_val,
        test_size=0.25,
        random_state=42
    )

    return train_split, validation_split, test_split


# Actually create the three datasets
train_split, validation_split, test_split = split_data(df)

print("Train shape:", train_split.shape)
print("Validation shape:", validation_split.shape)
print("Test shape:", test_split.shape)

Train shape: (1923, 12)
Validation shape: (642, 12)
Test shape: (642, 12)


## 6. Model Training, and Hyperparameter Tuning

- Train regression models on the train set (e.g., Ridge, Random Forest, XGBoost/LightGBM).
- Tune hyperparameters using the validation set.
- Use the `evaluate_predictions()` function (defined in Section 7) to report **train** and **validation**
  metrics for each model: RMSE, MAE, R².
- Select your best-performing model/configuration.

### Required output of this section
- **`predict_price(df)`** — takes a raw dataframe, calls `preprocess_data(df)` internally, and returns price predictions from your best trained model.

This is what Section 8 will call on the hidden test set, so both functions must work on data that has not been touched yet.

In [18]:
def predict_price(df):
    processed = preprocess_data(df)
    predictions = model.predict(processed)
    return predictions


from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor


X_train_raw = train_split.drop(columns=["price"])
y_train = train_split["price"]

X_val_raw = validation_split.drop(columns=["price"])
y_val = validation_split["price"]


X_train = preprocess_data(X_train_raw)
X_val = preprocess_data(X_val_raw)


numeric_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)


numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])


rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=300,
        max_depth=None,
        min_samples_split=2,
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)

rf_train_pred = rf_model.predict(X_train)
rf_val_pred = rf_model.predict(X_val)

print("Random Forest trained successfully.")


gb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ))
])

gb_model.fit(X_train, y_train)

gb_train_pred = gb_model.predict(X_train)
gb_val_pred = gb_model.predict(X_val)

print("Gradient Boosting trained successfully.")


# ------------------------------------------------------------
# 7. Temporary final model
# Change this after comparing validation MAE in Section 7
# ------------------------------------------------------------

model = rf_model


# ------------------------------------------------------------
# REQUIRED FUNCTION - used by Section 8
# ------------------------------------------------------------


Numeric features: ['model_year', 'milage', 'vehicle_age', 'horsepower', 'engine_size']
Categorical features: ['brand', 'model', 'fuel_type', 'engine', 'transmission', 'ext_col', 'int_col', 'accident', 'clean_title']


ValueError: could not convert string to float: '$52,000'

## 7. Evaluate on the Held-Out Test Set

Run `predict_price()` on your held-out test split (never used in training or tuning) and report
the same metrics using `evaluate_predictions()`.

In [14]:
def evaluate_predictions(y_true, y_pred):
    # DO NOT modify
    # Standard regression metric report used throughout this notebook.
    # Both your own evaluation (Section 6) and the grading step (Section 8) use this function.
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    metrics = {
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
    }
    print("--- Metrics ---")
    for name, value in metrics.items():
        print(f"{name:>10}: {value:.4f}")
    return metrics

---
## 8. Standardized Evaluation — DO NOT MODIFY

Everything below is run by the organizers for every submission. It loads the hidden evaluation set
and scores your submission. Make sure your notebook runs top-to-bottom.


In [ ]:
# =====================================================================
# ORGANIZER EVALUATION CELL — DO NOT MODIFY
# =====================================================================
assert "predict_price" in dir(), (
    "predict_price(df) is not defined. Your submission must define this "
    "function in Section 6 (see instructions there)."
)

evaluation_df = pd.read_csv(EVALUATION_SPLIT_PATH)
X_hidden = evaluation_df.drop(columns=[TARGET_COLUMN])
y_hidden = evaluation_df[TARGET_COLUMN]

y_pred = predict_price(X_hidden)

evaluation_metrics = evaluate_predictions(y_hidden, y_pred)
pd.DataFrame([evaluation_metrics])